# Genetic Wake-Word Search

End-to-end pipeline: dataset → genetic HP search → multi-tier training → ONNX export → benchmark.

## Configuration
Set env vars (Kaggle secrets / Paperspace env) or edit the **Config cell** below.

### Dataset modes (mutually exclusive, checked in order)
| Mode | Env vars set | Behaviour |
|------|--------------|-----------|
| **BYO CSV** | `CUSTOM_TRAIN_CSV` (+ optionally `CUSTOM_TEST_CSV`) | Use your own metadata CSVs — no TTS/HF downloads |
| **HF override** | `HF_DATASET` | Force a specific HuggingFace dataset ID for positives |
| **Auto** | _(neither)_ | Known wake words pull from HF; unknown synthesise via TTS |

### Augmentation & negative overrides (all optional, stackable)
| Variable | Format | Overrides |
|----------|--------|-----------|
| `NEGATIVES_DIR` | local path | Replace HF general negatives with a local audio directory |
| `EXTRA_NEGATIVES_HF` | `org/repo,...` | Append extra HF repos to general negatives |
| `BG_NOISE_DIR` | local path | Use this local directory for bg-noise augmentation |
| `EXTRA_BG_NOISE_HF` | `org/repo,...` | Append extra HF repos to bg-noise downloads |
| `MUSIC_DIR` | local path | Use this local directory for music augmentation |
| `EXTRA_MUSIC_HF` | `org/repo,...` | Append extra HF repos to music downloads |
| `RIR_DIR` | local path | Use this local directory for RIR augmentation |
| `EXTRA_RIR_HF` | `org/repo,...` | Append extra HF repos to RIR downloads |

Local-path overrides skip HF downloads for that category entirely.  
HF extras are downloaded **in addition to** the built-in repos.

### All variables
| Variable | Default | Description |
|----------|---------|-------------|
| `WAKE_WORD` | `hey jarvis` | Wake word phrase |
| `OUTPUT_DIR` | `./ww_output` | Root output directory |
| `LANG_CODE` | `en` | Language for TTS synthesis |
| `N_POSITIVE` | `200` | Positive samples to generate (ignored in BYO CSV mode) |
| `ADVERSARIAL` | `true` | Include adversarial negatives |
| `DOWNLOAD_AUGMENT` | `false` | Download HF bg-noise/music/RIR augmentation data |
| `CUSTOM_TRAIN_CSV` | _(empty)_ | Absolute path to your train metadata CSV (`path,label`) |
| `CUSTOM_TEST_CSV` | _(empty)_ | Absolute path to your test metadata CSV (optional) |
| `HF_DATASET` | _(empty)_ | HuggingFace dataset ID to use for positives |
| `NEGATIVES_DIR` | _(empty)_ | Local audio dir to use as general negatives (skips HF) |
| `EXTRA_NEGATIVES_HF` | _(empty)_ | Comma-separated HF repos appended to general negatives |
| `BG_NOISE_DIR` | _(empty)_ | Local audio dir for bg-noise augmentation (skips HF) |
| `EXTRA_BG_NOISE_HF` | _(empty)_ | Comma-separated HF repos appended to bg-noise |
| `MUSIC_DIR` | _(empty)_ | Local audio dir for music augmentation (skips HF) |
| `EXTRA_MUSIC_HF` | _(empty)_ | Comma-separated HF repos appended to music |
| `RIR_DIR` | _(empty)_ | Local audio dir for RIR augmentation (skips HF) |
| `EXTRA_RIR_HF` | _(empty)_ | Comma-separated HF repos appended to RIR |
| `POPULATION` | `12` | Genetic search population size |
| `GENERATIONS` | `5` | Number of generations |
| `EPOCHS_PER_TRIAL` | `3` | Epochs per genetic trial |
| `SEARCH_FULL` | `false` | Full search space (slower) |
| `TIERS_TO_TRAIN` | `micro,small,filterbank_small` | Comma-separated tiers for final training |
| `FINAL_EPOCHS` | `30` | Epochs for final model training |
| `EXPORT_ONNX` | `true` | Export ONNX models |
| `DEVICE` | `auto` | Device: auto, cpu, cuda, mps |
| `SEED` | `42` | Random seed |

In [ ]:
import os

WAKE_WORD         = os.environ.get("WAKE_WORD",         "hey jarvis")
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
LANG              = os.environ.get("LANG_CODE",         "en")
N_POSITIVE        = int(os.environ.get("N_POSITIVE",    "200"))
ADVERSARIAL       = os.environ.get("ADVERSARIAL",       "true").lower() == "true"
DOWNLOAD_AUGMENT  = os.environ.get("DOWNLOAD_AUGMENT",  "false").lower() == "true"
# Dataset overrides
CUSTOM_TRAIN_CSV  = os.environ.get("CUSTOM_TRAIN_CSV",  "")  # BYO train CSV
CUSTOM_TEST_CSV   = os.environ.get("CUSTOM_TEST_CSV",   "")  # BYO test CSV (optional)
HF_DATASET        = os.environ.get("HF_DATASET",        "")  # Force HF dataset for positives
# Negative / augmentation overrides
NEGATIVES_DIR     = os.environ.get("NEGATIVES_DIR",     "")  # local dir → general negatives
EXTRA_NEGATIVES_HF = os.environ.get("EXTRA_NEGATIVES_HF", "")  # extra HF repos, comma-sep
BG_NOISE_DIR      = os.environ.get("BG_NOISE_DIR",      "")  # local dir → bg-noise
EXTRA_BG_NOISE_HF = os.environ.get("EXTRA_BG_NOISE_HF", "")  # extra HF repos, comma-sep
MUSIC_DIR         = os.environ.get("MUSIC_DIR",         "")  # local dir → music
EXTRA_MUSIC_HF    = os.environ.get("EXTRA_MUSIC_HF",    "")  # extra HF repos, comma-sep
RIR_DIR           = os.environ.get("RIR_DIR",           "")  # local dir → RIR
EXTRA_RIR_HF      = os.environ.get("EXTRA_RIR_HF",      "")  # extra HF repos, comma-sep
# Genetic search
POPULATION        = int(os.environ.get("POPULATION",    "12"))
GENERATIONS       = int(os.environ.get("GENERATIONS",   "5"))
EPOCHS_PER_TRIAL  = int(os.environ.get("EPOCHS_PER_TRIAL", "3"))
SEARCH_FULL       = os.environ.get("SEARCH_FULL",       "false").lower() == "true"
# Final models
TIERS_TO_TRAIN    = os.environ.get("TIERS_TO_TRAIN",    "micro,small,filterbank_small").split(",")
FINAL_EPOCHS      = int(os.environ.get("FINAL_EPOCHS",  "30"))
EXPORT_ONNX       = os.environ.get("EXPORT_ONNX",       "true").lower() == "true"
DEVICE            = os.environ.get("DEVICE",            "auto")
SEED              = int(os.environ.get("SEED",          "42"))

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r}")

In [ ]:
import unittest.mock as _mock
from pathlib import Path
from ww_trainer.datagen import DatagenResult, download_hf_audio_dataset
import ww_trainer.datagen as _datagen_mod
from ww_trainer.quickstart import QuickstartConfig, _run_or_load_datagen


def _parse_hf_list(env_val: str) -> list[str]:
    """Parse a comma-separated list of HF repo IDs, stripping blanks."""
    return [s.strip() for s in env_val.split(",") if s.strip()]


def _build_neg_datasets_patch() -> dict:
    """Return a patched NEGATIVE_DATASETS that reflects any HF extras.

    Local-path overrides do not affect NEGATIVE_DATASETS because those
    categories are downloaded only when download_augmentation=True.
    Extra HF repos are appended to the existing lists so built-in repos
    are preserved unless the user explicitly replaces them.
    """
    import copy
    patched = copy.deepcopy(_datagen_mod.NEGATIVE_DATASETS)
    for key, env_val in [
        ("general",  EXTRA_NEGATIVES_HF),
        ("bg_noise", EXTRA_BG_NOISE_HF),
        ("music",    EXTRA_MUSIC_HF),
        ("rir",      EXTRA_RIR_HF),
    ]:
        extras = _parse_hf_list(env_val)
        if extras:
            patched[key] = patched.get(key, []) + extras
            print(f"  [{key}] extra HF repos: {extras}")
    return patched


# ── Mode 1: BYO CSV ──────────────────────────────────────────────────────────
if CUSTOM_TRAIN_CSV:
    train_path = Path(CUSTOM_TRAIN_CSV)
    assert train_path.exists(), f"CUSTOM_TRAIN_CSV not found: {train_path}"
    if CUSTOM_TEST_CSV:
        test_path = Path(CUSTOM_TEST_CSV)
        assert test_path.exists(), f"CUSTOM_TEST_CSV not found: {test_path}"
    else:
        import random as _rnd
        _rnd.seed(SEED)
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        train_path_split = split_dir / "train_metadata.csv"
        test_path = split_dir / "test_metadata.csv"
        if not (train_path_split.exists() and test_path.exists()):
            rows = train_path.read_text().splitlines()
            _rnd.shuffle(rows)
            cut = int(len(rows) * 0.8)
            train_path_split.write_text("\n".join(rows[:cut]))
            test_path.write_text("\n".join(rows[cut:]))
            print(f"Split {len(rows)} rows → train={cut}, test={len(rows)-cut}")
        else:
            print("Reusing existing split (resume)")
        train_path = train_path_split
    datagen_result = DatagenResult(
        train_csv=train_path,
        test_csv=test_path,
        positives_dir=train_path.parent,
        negatives_dir=Path(NEGATIVES_DIR) if NEGATIVES_DIR else train_path.parent,
        bg_noise_dir=Path(BG_NOISE_DIR) if BG_NOISE_DIR else None,
        music_dir=Path(MUSIC_DIR)     if MUSIC_DIR    else None,
        rir_dir=Path(RIR_DIR)         if RIR_DIR      else None,
    )
    print(f"Mode: BYO CSV | train={train_path} | test={test_path}")

# ── Modes 2 & 3: HF override or Auto (share datagen path) ───────────────────
else:
    # Patch NEGATIVE_DATASETS if extra HF repos were requested
    patched_neg = _build_neg_datasets_patch()

    # Optionally patch find_positive_dataset for HF override mode
    pos_patch = (
        _mock.patch.object(_datagen_mod, "find_positive_dataset", return_value=HF_DATASET)
        if HF_DATASET else _mock.patch.object(_datagen_mod, "find_positive_dataset",
                                               side_effect=_datagen_mod.find_positive_dataset)
    )

    with _mock.patch.object(_datagen_mod, "NEGATIVE_DATASETS", patched_neg), pos_patch:
        cfg = QuickstartConfig(
            wake_word=WAKE_WORD, output_dir=Path(OUTPUT_DIR),
            n_positive=N_POSITIVE, lang=LANG,
            adversarial=ADVERSARIAL, download_augmentation=DOWNLOAD_AUGMENT,
            reuse_dataset=True, seed=SEED,
        )
        datagen_result = _run_or_load_datagen(cfg)

    mode = f"HF override ({HF_DATASET})" if HF_DATASET else "Auto"
    print(f"Mode: {mode}")

    # Apply local-path overrides to the result (skip HF downloads for those categories)
    if NEGATIVES_DIR:
        datagen_result.negatives_dir = Path(NEGATIVES_DIR)
        print(f"  negatives_dir overridden → {NEGATIVES_DIR}")
    if BG_NOISE_DIR:
        datagen_result.bg_noise_dir = Path(BG_NOISE_DIR)
        print(f"  bg_noise_dir overridden  → {BG_NOISE_DIR}")
    if MUSIC_DIR:
        datagen_result.music_dir = Path(MUSIC_DIR)
        print(f"  music_dir overridden     → {MUSIC_DIR}")
    if RIR_DIR:
        datagen_result.rir_dir = Path(RIR_DIR)
        print(f"  rir_dir overridden       → {RIR_DIR}")

    # Download any extra HF augmentation repos not covered by datagen
    aug_dir = Path(OUTPUT_DIR) / "augmentation"
    for env_val, sub, field in [
        (EXTRA_BG_NOISE_HF, "bg_noise", "bg_noise_dir"),
        (EXTRA_MUSIC_HF,    "music",    "music_dir"),
        (EXTRA_RIR_HF,      "rir",      "rir_dir"),
    ]:
        extras = _parse_hf_list(env_val)
        if extras and not DOWNLOAD_AUGMENT:
            # datagen skipped HF aug downloads; fetch extras manually
            dest = aug_dir / sub
            dest.mkdir(parents=True, exist_ok=True)
            for ds_id in extras:
                download_hf_audio_dataset(ds_id, dest / ds_id.split("/")[-1])
            if not getattr(datagen_result, field):
                setattr(datagen_result, field, dest)

n_train = sum(1 for _ in open(datagen_result.train_csv))
n_test  = sum(1 for _ in open(datagen_result.test_csv))
print(f"Train: {n_train} samples | Test: {n_test} samples")

In [ ]:
from ww_trainer.sweep import run_genetic_search

genetic_result = run_genetic_search(
    metadata_csv=str(datagen_result.train_csv),
    population_size=POPULATION,
    generations=GENERATIONS,
    epochs_per_trial=EPOCHS_PER_TRIAL,
    featurizer_type="mfcc",
    device=DEVICE,
    output_dir=str(Path(OUTPUT_DIR) / "genetic"),
    full=SEARCH_FULL,
)
best_hp  = genetic_result["best_config"]
best_f1  = genetic_result["best_score"]
print(f"Best search F1 : {best_f1:.4f}")
print(f"Best config    : {best_hp}")

In [ ]:
import matplotlib.pyplot as plt

history = genetic_result["history"]
gens  = [h["generation"] for h in history]
bests = [h["best"]       for h in history]
avgs  = [h["avg"]        for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, bests, "o-",  label="Best F1")
ax.plot(gens, avgs,  "s--", label="Avg F1",  alpha=0.7)
ax.set_xlabel("Generation"); ax.set_ylabel("F1 (search)")
ax.set_title(f"Genetic Search Evolution \u2014 {WAKE_WORD!r}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "evolution.png", dpi=120)
plt.show()

In [ ]:
from ww_trainer.quickstart import _train_from_datagen_result

final_results = []
for tier_name in TIERS_TO_TRAIN:
    model_out = Path(OUTPUT_DIR) / f"model_{tier_name}"
    tier_cfg = QuickstartConfig(
        wake_word=WAKE_WORD, output_dir=model_out,
        tier=tier_name,
        epochs=FINAL_EPOCHS,
        batch_size=best_hp.get("batch_size", 16),
        lr=best_hp.get("lr", 5e-4),
        export_onnx=EXPORT_ONNX,
        device=DEVICE,
        download_augmentation=False,
        seed=SEED,
    )
    result = _train_from_datagen_result(tier_cfg, datagen_result)
    final_results.append({
        "tier": tier_name,
        "f1":   result.metrics.get("f1", 0.0),
        "onnx": result.best_onnx_path,
        "pt":   result.best_model_path,
    })
    print(f"  {tier_name:20s}  F1={result.metrics.get('f1', 0):.3f}")

In [ ]:
names  = [r["tier"] for r in final_results]
scores = [r["f1"]   for r in final_results]
colors = plt.cm.tab10.colors[:len(names)]

fig, ax = plt.subplots(figsize=(max(5, len(names)*1.8), 4))
bars = ax.bar(names, scores, color=colors)
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
ax.set_ylim(0, 1.1); ax.set_ylabel("F1")
ax.set_title(f"Tier Comparison \u2014 {WAKE_WORD!r}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "tier_comparison.png", dpi=120)
plt.show()

In [ ]:
from ww_trainer.benchmark import run_benchmark, plot_results, save_results
from IPython.display import Image, display

bench_dir = Path(OUTPUT_DIR) / "benchmark"
bench_dir.mkdir(parents=True, exist_ok=True)
bench_report = run_benchmark(
    device=DEVICE,
    output_dir=str(bench_dir),
)
save_results(bench_report, bench_dir)
plot_results(bench_report, bench_dir)

for png in sorted(bench_dir.glob("*.png")):
    display(Image(str(png)))

In [ ]:
from IPython.display import Markdown, display

rows = ["| Tier | F1 | ONNX |", "|------|-----|------|"] 
for r in final_results:
    onnx_mark = "\u2713" if r["onnx"] and Path(r["onnx"]).exists() else "\u2014"
    rows.append(f"| {r['tier']} | {r['f1']:.3f} | {onnx_mark} |")
display(Markdown("\n".join(rows)))

print(f"\nAll outputs saved to: {Path(OUTPUT_DIR).resolve()}")

## Next Steps

- **Resume after crash**: re-run from Cell 4 — dataset is reused, no TTS re-synthesis
- **BYO dataset**: set `CUSTOM_TRAIN_CSV=/path/to/metadata.csv` (format: `path,label`)
- **BYO negatives**: set `NEGATIVES_DIR=/path/to/neg_audio/` to skip HF negative downloads
- **BYO augmentation**: set `BG_NOISE_DIR`, `MUSIC_DIR`, and/or `RIR_DIR` to local dirs
- **Extra HF data**: set `EXTRA_NEGATIVES_HF=org/repo1,org/repo2` (or the per-category variants)
- **Specific HF positives**: set `HF_DATASET=org/repo-name` to override auto-detection
- **More tiers**: set `TIERS_TO_TRAIN=micro,small,medium,large`
- **Deeper search**: set `SEARCH_FULL=true`, increase `POPULATION` and `GENERATIONS`
- **Full dataset**: set `N_POSITIVE=500`, `DOWNLOAD_AUGMENT=true` for production models

### Resources
- [Quickstart guide](../docs/quickstart.md)
- [Training docs](../docs/training.md)
- [Hardware guide](../docs/hardware_guide.md)
- [All tiers reference](../docs/classifiers.md)
- [Search strategies](../docs/search_strategies.md)